In [1]:
import pandas as pd

In [25]:
import pandas as pd

# --- Ler o dataset original ---
df = pd.read_csv("128_fishing_trajs.csv")

# --- Obter os 10 primeiros trajectory_id ---
first_ids = sorted(df['id'].unique())[:1]

# --- Filtrar apenas essas trajetórias ---
df_filtered = df[df['id'].isin(first_ids)]

# --- Criar o novo dataframe no formato desejado ---
df_new = (
    df_filtered.groupby('id', group_keys=False)
      .apply(lambda g: pd.DataFrame({
          'trajectory_id': g['id'].values,
          'timestamp': range(len(g)),
          'x': g['x'].values,
          'y': g['y'].values,
          'bearing': g['bearing'].values,
          'label': g['label'].values
      }))
      .reset_index(drop=True)
)

# --- Exibir resultado ---
print("IDs selecionados:", first_ids)
print(df_new.head())
print("\nFormato final:", df_new.shape)
df_new.to_csv("128_fishing_trajsRugs10.csv")

          
 

IDs selecionados: ['211477000-2']
  trajectory_id  timestamp              x             y     bearing  \
0   211477000-2          0  705723.070725  6.225237e+06  309.501333   
1   211477000-2          1  705720.961925  6.225238e+06  300.882654   
2   211477000-2          2  705718.785662  6.225240e+06  302.344350   
3   211477000-2          3  705716.469132  6.225241e+06  306.781649   
4   211477000-2          4  705713.779868  6.225243e+06  302.651628   

        label  
0  01-sailing  
1  01-sailing  
2  01-sailing  
3  01-sailing  
4  01-sailing  

Formato final: (5610, 6)


C:\Users\ccana\AppData\Local\Temp\ipykernel_27088\3445370990.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.DataFrame({


In [16]:
# Célula: detectar todos os segmentos "sailing" por trajectory_id (robusta contra erro de índice)
import pandas as pd

# carregar CSV se necessário:
# df = pd.read_csv("seu_arquivo.csv")

# garantias básicas
df = df.copy()
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df['x'] = pd.to_numeric(df['x'], errors='coerce')
df['y'] = pd.to_numeric(df['y'], errors='coerce')

# ordenar por trajectory_id + timestamp (muito importante)
df = df.sort_values(['trajectory_id', 'timestamp']).reset_index(drop=True)

# máscara de sailing (insensível a maiúsc/minúsc)
df['is_sailing'] = df['label'].astype(str).str.contains('sailing', case=False, na=False)

if not df['is_sailing'].any():
    print("Nenhum trecho com 'sailing' encontrado.")
else:
    # detectar início de cada segmento sailing:
    # início quando is_sailing == True e (ou veio de outro trajectory_id) ou (ou veio de is_sailing False)
    prev_traj = df['trajectory_id'].shift()
    prev_sailing = df['is_sailing'].shift().fillna(False)

    df['start_mask'] = (
        (df['is_sailing']) &
        (
            (df['trajectory_id'] != prev_traj) |  # novo trajectory_id -> conta como novo segmento se for sailing
            (~prev_sailing)                       # vinha de False -> é início
        )
    )

    # enumerar os segmentos por trajectory_id com cumsum (mantém index alinhado)
    df['segment_run'] = df.groupby('trajectory_id')['start_mask'].cumsum()

    # queremos apenas os pontos onde is_sailing == True e segment_run > 0
    sailing_points = df[df['is_sailing'] & (df['segment_run'] > 0)].copy()

    # agora agregar por trajectory_id + segment_run para obter início/fim do segmento
    segments = (sailing_points
                .groupby(['trajectory_id', 'segment_run'], sort=True)
                .agg(
                    segment_id=('segment_run', 'first'),    # id do segmento dentro da trajetória
                    start_time=('timestamp', 'first'),
                    start_x=('x', 'first'),
                    start_y=('y', 'first'),
                    end_time=('timestamp', 'last'),
                    end_x=('x', 'last'),
                    end_y=('y', 'last'),
                    n_points=('timestamp', 'size')
                )
                .reset_index(drop=False)
               )

    # renomear/organizar colunas: segment_id começa em 1 por trajectory_id (opcional)
    segments['segment_index_within_traj'] = segments.groupby('trajectory_id').cumcount() + 1

    # calcular duração
    segments['duration'] = segments['end_time'] - segments['start_time']

    # ordenar e mostrar
    segments = segments[[
        'trajectory_id', 'segment_index_within_traj', 'segment_id',
        'start_time','end_time','duration','n_points',
        'start_x','start_y','end_x','end_y'
    ]].sort_values(['trajectory_id','segment_index_within_traj']).reset_index(drop=True)

    display(segments)   # Jupyter-friendly
    print(segments.to_string(index=False))

    # salvar
    #segments.to_csv('sailing_segments_per_trajectory.csv', index=False)


C:\Users\ccana\AppData\Local\Temp\ipykernel_27088\1604617921.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev_sailing = df['is_sailing'].shift().fillna(False)


,trajectory_id,segment_index_within_traj,segment_id,start_time,end_time,duration,n_points,start_x,start_y,end_x,end_y
0,211477000-2,1,1,0,389,389,390,705723.070725,6.225237e+06,706461.491500,6.235998e+06
1,211477000-2,2,2,5063,5609,546,547,705431.435088,6.235937e+06,705755.234613,6.225180e+06
2,211477000-3,1,1,0,270,270,271,705762.672600,6.225219e+06,706135.778763,6.235860e+06
3,211477000-3,2,2,4861,5474,613,614,706343.089960,6.235867e+06,705759.439166,6.225193e+06
4,211477000-4,1,1,0,290,290,291,705769.562149,6.225213e+06,706210.760373,6.236105e+06
5,211477000-4,2,2,4844,5377,533,534,705011.524488,6.236791e+06,705751.860907,6.225199e+06
6,219000855-3,1,1,0,441,441,442,628079.562928,6.355216e+06,639382.579630,6.365148e+06
7,219000855-3,2,2,4253,5038,785,786,641860.259487,6.366537e+06,628078.715036,6.355201e+06
8,219000855-4,1,1,0,444,444,445,628083.861850,6.355216e+06,639889.755013,6.365017e+06
9,219000855-4,2,2,3762,4668,906,907,640671.735183,6.365958e+06,628084.329713,6.355191e+06


trajectory_id  segment_index_within_traj  segment_id  start_time  end_time  duration  n_points       start_x      start_y         end_x        end_y
  211477000-2                          1           1           0       389       389       390 705723.070725 6.225237e+06 706461.491500 6.235998e+06
  211477000-2                          2           2        5063      5609       546       547 705431.435088 6.235937e+06 705755.234613 6.225180e+06
  211477000-3                          1           1           0       270       270       271 705762.672600 6.225219e+06 706135.778763 6.235860e+06
  211477000-3                          2           2        4861      5474       613       614 706343.089960 6.235867e+06 705759.439166 6.225193e+06
  211477000-4                          1           1           0       290       290       291 705769.562149 6.225213e+06 706210.760373 6.236105e+06
  211477000-4                          2           2        4844      5377       533       534 705011.5244

In [17]:
# Célula: detectar todos os segmentos "sailing" por trajectory_id (robusta contra erro de índice)
import pandas as pd

# carregar CSV se necessário:
# df = pd.read_csv("seu_arquivo.csv")

# garantias básicas
df = df.copy()
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df['x'] = pd.to_numeric(df['x'], errors='coerce')
df['y'] = pd.to_numeric(df['y'], errors='coerce')

# ordenar por trajectory_id + timestamp (muito importante)
df = df.sort_values(['trajectory_id', 'timestamp']).reset_index(drop=True)

# máscara de sailing (insensível a maiúsc/minúsc)
df['is_sailing'] = df['label'].astype(str).str.contains('fishing', case=False, na=False)

if not df['is_sailing'].any():
    print("Nenhum trecho com 'sailing' encontrado.")
else:
    # detectar início de cada segmento sailing:
    # início quando is_sailing == True e (ou veio de outro trajectory_id) ou (ou veio de is_sailing False)
    prev_traj = df['trajectory_id'].shift()
    prev_sailing = df['is_sailing'].shift().fillna(False)

    df['start_mask'] = (
        (df['is_sailing']) &
        (
            (df['trajectory_id'] != prev_traj) |  # novo trajectory_id -> conta como novo segmento se for sailing
            (~prev_sailing)                       # vinha de False -> é início
        )
    )

    # enumerar os segmentos por trajectory_id com cumsum (mantém index alinhado)
    df['segment_run'] = df.groupby('trajectory_id')['start_mask'].cumsum()

    # queremos apenas os pontos onde is_sailing == True e segment_run > 0
    sailing_points = df[df['is_sailing'] & (df['segment_run'] > 0)].copy()

    # agora agregar por trajectory_id + segment_run para obter início/fim do segmento
    segments = (sailing_points
                .groupby(['trajectory_id', 'segment_run'], sort=True)
                .agg(
                    segment_id=('segment_run', 'first'),    # id do segmento dentro da trajetória
                    start_time=('timestamp', 'first'),
                    start_x=('x', 'first'),
                    start_y=('y', 'first'),
                    end_time=('timestamp', 'last'),
                    end_x=('x', 'last'),
                    end_y=('y', 'last'),
                    n_points=('timestamp', 'size')
                )
                .reset_index(drop=False)
               )

    # renomear/organizar colunas: segment_id começa em 1 por trajectory_id (opcional)
    segments['segment_index_within_traj'] = segments.groupby('trajectory_id').cumcount() + 1

    # calcular duração
    segments['duration'] = segments['end_time'] - segments['start_time']

    # ordenar e mostrar
    segments = segments[[
        'trajectory_id', 'segment_index_within_traj', 'segment_id',
        'start_time','end_time','duration','n_points',
        'start_x','start_y','end_x','end_y'
    ]].sort_values(['trajectory_id','segment_index_within_traj']).reset_index(drop=True)

    display(segments)   # Jupyter-friendly
    print(segments.to_string(index=False))

    # salvar
    #segments.to_csv('sailing_segments_per_trajectory.csv', index=False)


C:\Users\ccana\AppData\Local\Temp\ipykernel_27088\2703292870.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev_sailing = df['is_sailing'].shift().fillna(False)


,trajectory_id,segment_index_within_traj,segment_id,start_time,end_time,duration,n_points,start_x,start_y,end_x,end_y
0,211477000-2,1,1,390,5062,4672,4673,706460.597481,6.236040e+06,705429.566640,6.235967e+06
1,211477000-3,1,1,271,4860,4589,4590,706134.906352,6.235899e+06,706338.178478,6.235894e+06
2,211477000-4,1,1,291,4843,4552,4553,706209.510658,6.236108e+06,705005.994993,6.236813e+06
3,219000855-3,1,1,442,4252,3810,3811,639407.591903,6.365157e+06,641860.342566,6.366533e+06
4,219000855-4,1,1,445,3761,3316,3317,639909.696095,6.365025e+06,640686.365617,6.365958e+06
5,219001039-2,1,1,1187,5150,3963,3964,445081.912491,6.172116e+06,443883.420092,6.169409e+06
6,219001125-3,1,1,1041,4348,3307,3308,581637.156103,6.425049e+06,582090.008897,6.425745e+06
7,219001149-2,1,1,734,9809,9075,9076,640226.319516,6.344634e+06,639323.664016,6.348990e+06
8,219001149-2,2,2,11494,22626,11132,11133,640595.729399,6.345684e+06,642083.727819,6.347500e+06
9,219001522-2,1,1,1859,4882,3023,3024,578299.698145,6.432035e+06,578251.018618,6.431986e+06


trajectory_id  segment_index_within_traj  segment_id  start_time  end_time  duration  n_points       start_x      start_y         end_x        end_y
  211477000-2                          1           1         390      5062      4672      4673 706460.597481 6.236040e+06 705429.566640 6.235967e+06
  211477000-3                          1           1         271      4860      4589      4590 706134.906352 6.235899e+06 706338.178478 6.235894e+06
  211477000-4                          1           1         291      4843      4552      4553 706209.510658 6.236108e+06 705005.994993 6.236813e+06
  219000855-3                          1           1         442      4252      3810      3811 639407.591903 6.365157e+06 641860.342566 6.366533e+06
  219000855-4                          1           1         445      3761      3316      3317 639909.696095 6.365025e+06 640686.365617 6.365958e+06
  219001039-2                          1           1        1187      5150      3963      3964 445081.9124